# Statistical comparison of the models

Friedman test, average ranks, 95% confidence intervals of the 10-fold means, and two-sided Wilcoxon signed-rank tests between XGBoost and each other model with Holm correction.

Run `ml_models.ipynb` and `dl_models.ipynb` first. This notebook uses their fold results.

In [ ]:
import os

import numpy as np
import pandas as pd
from scipy import stats

OUT_DIR = "results"
REF = "XGBoost"
N_FOLDS = 10

ml = pd.read_csv(os.path.join(OUT_DIR, "ml_cv_results_folds.csv"))
dl = pd.read_csv(os.path.join(OUT_DIR, "dl_cv_results_folds.csv"))
folds = pd.concat([ml[["model", "fold", "accuracy", "f1_macro"]], dl[["model", "fold", "accuracy", "f1_macro"]]],
                  ignore_index=True).astype({"fold": int, "accuracy": float, "f1_macro": float})


def holm(pvalues):
    p = np.asarray(pvalues, dtype=float)
    order = np.argsort(p)
    adjusted = np.empty(len(p))
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, min(1.0, (len(p) - rank) * p[idx]))
        adjusted[idx] = running
    return adjusted

In [ ]:
summary_rows, test_rows = [], []
for metric, label in [("accuracy", "Accuracy"), ("f1_macro", "Macro F1")]:
    P = folds.pivot(index="fold", columns="model", values=metric)
    friedman = stats.friedmanchisquare(*[P[c] for c in P.columns])
    print(f"{label}: Friedman chi2 = {friedman.statistic:.2f}, p = {friedman.pvalue:.3g}")

    ranks = P.rank(axis=1, ascending=False).mean()
    t_crit = stats.t.ppf(0.975, N_FOLDS - 1)
    for m in P.columns:
        v = P[m]
        half = t_crit * v.std(ddof=1) / np.sqrt(N_FOLDS)
        summary_rows.append(dict(metric=label, model=m, mean=v.mean(), ci_low=v.mean() - half,
                                 ci_high=v.mean() + half, avg_rank=ranks[m]))

    rows = []
    for m in P.columns:
        if m == REF:
            continue
        diff = P[REF] - P[m]
        w = stats.wilcoxon(P[REF], P[m], alternative="two-sided", zero_method="wilcox",
                           method="exact" if (diff != 0).all() else "auto")
        rows.append(dict(metric=label, comparison=f"{REF} vs {m}", mean_difference=diff.mean(),
                         wilcoxon_p=w.pvalue))
    rows = pd.DataFrame(rows)
    rows["p_holm"] = holm(rows["wilcoxon_p"])
    test_rows.append(rows)

summary = pd.DataFrame(summary_rows)
tests = pd.concat(test_rows, ignore_index=True)
summary.to_csv(os.path.join(OUT_DIR, "model_mean_ci_rank.csv"), index=False)
tests.to_csv(os.path.join(OUT_DIR, "wilcoxon_holm_xgboost_vs_others.csv"), index=False)
print(summary.sort_values(["metric", "avg_rank"]).round(4).to_string(index=False))
print(tests.round(4).to_string(index=False))